# EduGuard-AI — Teacher Login + Optional LLM (Separate Colab Notebook)
This is a new experimental version. The original `app.py`, `train.py`, and `EduGuard-AI-GitHub-Colab.ipynb` are unchanged.

**Functions:** teacher login, test performance, student risk probability, SHAP explanation, conformal prediction sets, optional LLM support guidance, and batch CSV prediction.

**Safety:** Colab sessions are temporary; use only anonymous demonstration/student test data. The LLM only receives a numeric summary after consent. **Do not publish credentials or API keys.**

## 1. Configure Colab Secrets
Click the **key icon** in the left Colab sidebar (Secrets). Add these names:
- `EDUGUARD_TEACHER_USER`: username, e.g., your chosen teacher username.
- `EDUGUARD_TEACHER_PASSWORD`: a **new strong password with at least 12 characters** (do not share or paste it into code).
- `OPENAI_API_KEY`: optional API key for LLM explanations. Leave unset if you want the login + prediction system without the LLM.
- `EDUGUARD_LLM_MODEL`: optional model ID. Defaults to `gpt-4.1-mini`.

Enable **Notebook access** for the secrets. An API key requires a provider account and may incur usage charges. The API is not the same as your ChatGPT login/subscription.

In [ ]:
%cd /content
!git clone https://github.com/cseteesta-bit/EduGuard-AI.git EduGuard-AI-Teacher-LLM
%cd /content/EduGuard-AI-Teacher-LLM

In [ ]:
%pip -q install -r requirements-teacher-llm.txt

## 2. Load credentials (not printed or stored in the notebook)
The notebook reads values from Colab Secrets and transfers them to the Streamlit process as environment variables. No passwords or API keys are written to the repository.

In [ ]:
from google.colab import userdata
import os
def read_secret(name, required=False):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f"Missing Colab Secret: {name}. Set it in the Secrets sidebar and enable notebook access.")
    return value or ""
os.environ["EDUGUARD_TEACHER_USER"] = read_secret("EDUGUARD_TEACHER_USER", required=True)
os.environ["EDUGUARD_TEACHER_PASSWORD"] = read_secret("EDUGUARD_TEACHER_PASSWORD", required=True)
if len(os.environ["EDUGUARD_TEACHER_PASSWORD"]) < 12:
    raise ValueError("EDUGUARD_TEACHER_PASSWORD must have at least 12 characters.")
key = read_secret("OPENAI_API_KEY")
if key:
    os.environ["OPENAI_API_KEY"] = key
model_name = read_secret("EDUGUARD_LLM_MODEL")
if model_name:
    os.environ["EDUGUARD_LLM_MODEL"] = model_name
print("Teacher credentials loaded. LLM:", "Enabled" if key else "Disabled (optional)")

## 3. Obtain OULAD and train the original ML pipeline
This reuses the **unchanged** project's model training code. The full public OULAD dataset is downloaded from UCI into this Colab workspace. The resulting trained model is local to this runtime.

In [ ]:
from pathlib import Path
import urllib.request, zipfile, shutil
root = Path("data/oulad")
root.mkdir(parents=True, exist_ok=True)
required = ["studentInfo.csv", "studentVle.csv", "studentAssessment.csv", "assessments.csv"]
if not all((root/name).is_file() for name in required):
    archive = Path("/content/oulad_teacher_original.zip")
    url = "https://archive.ics.uci.edu/static/public/349/open%2Buniversity%2Blearning%2Banalytics%2Bdataset.zip"
    urllib.request.urlretrieve(url, archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(root)
    for nested in list(root.rglob("*.zip")):
        with zipfile.ZipFile(nested) as z:
            z.extractall(nested.parent)
    for name in required:
        if (root/name).is_file():
            continue
        found = [p for p in root.rglob(name) if p.is_file()]
        if not found:
            raise FileNotFoundError(f"Missing OULAD file: {name}")
        shutil.copy2(found[0],root/name)
    archive.unlink(missing_ok=True)
print("OULAD ready:", {name: (root/name).exists() for name in required})

In [ ]:
import subprocess, json
from pathlib import Path
subprocess.run(["python", "train.py", "--data-dir", "data/oulad"], check=True)
data = json.loads(Path("models/metrics.json").read_text())
print("Selected model:", data["selected_model"])
for name, item in data["models"].items():
    m = item["test"]
    print(name, {k: round(m[k],4) if m.get(k) is not None else None for k in
                 ["accuracy", "f1", "roc_auc", "coverage", "avg_set_size"]})

## 4. Launch the new teacher portal
The username and password prompt is shown inside the Streamlit app. The Colab reverse proxy requires relaxed origin protection to display the WebSocket. **This is only suitable for a temporary, non-sensitive classroom demonstration**, not production hosting.

In [ ]:
import subprocess, time, requests
from google.colab import output
subprocess.run(["pkill", "-f", "[s]treamlit run app_teacher_llm.py"], check=False)
logfile = open("/tmp/eduguard_teacher_portal.log", "w")
server = subprocess.Popen([
    "streamlit", "run", "app_teacher_llm.py",
    "--server.port", "8502",
    "--server.address", "0.0.0.0",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
], stdout=logfile, stderr=subprocess.STDOUT, env=os.environ.copy())
time.sleep(6)
if server.poll() is not None:
    print(Path("/tmp/eduguard_teacher_portal.log").read_text()[-3000:])
    raise RuntimeError("Teacher portal did not start.")
res = requests.get("http://127.0.0.1:8502/_stcore/health", timeout=10)
print("Streamlit health:", res.status_code, res.text)
print("Open teacher portal:", output.eval_js("google.colab.kernel.proxyPort(8502)"))
display(output.serve_kernel_port_as_iframe(8502,height=800))

## 5. How to use the LLM feature
1. Sign in using the teacher credentials in Colab Secrets.
2. Open **Student Risk + LLM**.
3. Select a sample and click **Analyze student risk**.
4. Review risk, conformal label set, feature contributions, and rule-based support suggestions.
5. If you supplied `OPENAI_API_KEY`, check the consent box and click **Generate LLM support explanation**.
6. Verify the generated suggestions; they are not decisions or causal findings.

LLM usage sends **only** a limited de-identified numeric summary and up to five model feature contributions. Names, IDs, module identifiers, and CSV files are not sent. For true institutional usage, get privacy, security and ethics approval and replace the single shared login with a professional identity provider.